In [1]:
!pip install yfinance
import pandas as pd
import yfinance as yf
import requests
!pip install lxml

mediummomentum = pd.read_csv("C:\\Users\\henry\\OneDrive\\Documents\\GitHub\\Cyclops\\data\\mediummomentum.csv", index_col=0, parse_dates=True)
returns_m = pd.read_csv("C:\\Users\\henry\\OneDrive\\Documents\\GitHub\\Cyclops\\data\\returns_m.csv", index_col=0, parse_dates=True)


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
top_list = []
bottom_list = []

for k in range(mediummomentum.shape[0]):
    row = mediummomentum.iloc[k]
    valid = row.dropna()               
    n = max(1, len(valid) // 10) # finding number of stocks in top and bottom 10%.
    order = valid.sort_values() 
    bottom_list.append(order.index[:n].tolist()) # appending the bottom decile tickers to the bottom_list.
    top_list.append(order.index[-n:].tolist()) # appending the top decile tickers to the top_list.

mediummomentum["top_decile"] = top_list
mediummomentum["bottom_decile"] = bottom_list

C:\Users\henry\AppData\Local\Temp\ipykernel_33224\3599614039.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mediummomentum["top_decile"] = top_list
C:\Users\henry\AppData\Local\Temp\ipykernel_33224\3599614039.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mediummomentum["bottom_decile"] = bottom_list


In [4]:
stock_cols = [c for c in mediummomentum.columns if c.startswith('12M Momentum for')]
pos = {c: i for i, c in enumerate(stock_cols)} 

mediummomentum['top_decile'] = mediummomentum['top_decile'].apply(lambda lst: [pos[c] for c in lst])
mediummomentum['bottom_decile'] = mediummomentum['bottom_decile'].apply(lambda lst: [pos[c] for c in lst]) # map the tickers to their column indices

In [5]:
def portfolio(k):
    top_returns = []
    bottom_returns = []

    for i in range(mediummomentum.shape[0]):
        top = mediummomentum["top_decile"].iloc[i]
        bottom = mediummomentum["bottom_decile"].iloc[i] # get the top and bottom decile tickers for the current month

        if i + k > returns_m.shape[0] or len(top) == 0 or len(bottom) == 0:
            top_returns.append(None)
            bottom_returns.append(None) # if there are not enough months of data to calculate the returns, or if there are no stocks in the top or bottom decile, we append None to the returns lists.
        else:
            total = 0
            for j in range(len(top)):
                growth = 1
                for o in range(k):
                    growth *= (1 + returns_m.iloc[i+o, top[j]]) # calculate the growth of each stock in the top decile over the next k months. 
                total = total + growth -1
            top_returns.append(total/len(top)) # averaging returns of all stocks in the top decile. 

            total = 0
            for j in range(len(bottom)):
                growth = 1
                for o in range(k):
                    growth = growth * (1 + returns_m.iloc[i + o, bottom[j]]) # calculate the growth of each stock in the bottom decile over the next k months. 
                total = total + (growth - 1)
            bottom_returns.append(total / len(bottom)) # averaging returns of all stocks in the bottom decile. 

    results = pd.DataFrame(index = mediummomentum.index)
    results["top"] = top_returns
    results["bottom"] = bottom_returns
    results["long_short"] = results["top"] - results["bottom"] # buying top deciles and shorting bottom deciles, this column shows the long-short portfolio returns. 
    return results


portfolio(1).head(25)
 


,top,bottom,long_short
Date,,,
2010-01-31,NaN,NaN,NaN
2010-02-28,NaN,NaN,NaN
2010-03-31,NaN,NaN,NaN
2010-04-30,NaN,NaN,NaN
2010-05-31,NaN,NaN,NaN
2010-06-30,NaN,NaN,NaN
2010-07-31,NaN,NaN,NaN
2010-08-31,NaN,NaN,NaN
2010-09-30,NaN,NaN,NaN


In [7]:
mediummomentum.head()
#we want to see the portfolio returns for a holding period of k months. If we set k at e.g. 3, then we buy the top decile and short the bottom decile of each row for 3 months. However, this means that at any given month (after the first 3 months), we are essentially holding 3 different portfolios, where the first portfolio will be held for 3 months, the second at its 2 month mark and the most recent portfolio for 1 month. That is our 'portfolio return', which is the average of all 3 portfolios. 



def strategy(k):
    portfolio_returns = []

    for j in range(mediummomentum.shape[0]):
        batch_returns = []

        for i in range(k):
            c = j - i   # seeing months in the past to gather medium momentum data for the current month.                   

            if c >= 0:                   
                top = mediummomentum["top_decile"].iloc[c] # get the top and bottom decile tickers for the current month
                bottom = mediummomentum["bottom_decile"].iloc[c]

                if len(top) > 0 and len(bottom) > 0: # condition to ensure that there are stocks in both the top and bottom deciles.
                    top_total = 0
                    for a in range(len(top)):
                        top_total = top_total + returns_m.iloc[j, top[a]] # calculate the total returns for the top decile stocks for the month in the past. 
                    bottom_total = 0
                    for a in range(len(bottom)):
                        bottom_total = bottom_total + returns_m.iloc[j, bottom[a]]

                    batch_returns.append(top_total / len(top) - bottom_total / len(bottom)) # calculate the portfolio return for the month should we buy the top stocks and short the bottom ones. 

        if len(batch_returns) < k:      
            portfolio_returns.append(None)
        else:
            portfolio_returns.append(sum(batch_returns) / k) # when k months of data is available, we average the portfolio returns across all k months to get the portfolio return for the current month.

    results = pd.DataFrame(index=mediummomentum.index)
    results["portfolio_return"] = portfolio_returns
    return results

threemonthhold = strategy(3)
onemonthhold = strategy(1)
sixmonthhold = strategy(6)
twlevemonthhold = strategy(12)

portfolio_returns = pd.concat([onemonthhold, threemonthhold, sixmonthhold, twlevemonthhold], axis=1)
portfolio_returns.columns = ["1 Month", "3 Months", "6 Months", "12 Months"] # trying to see the portfolio returns for different holding periods to optimise k.

print(portfolio_returns.head(25))
portfolio_returns.to_csv("C:\\Users\\henry\\OneDrive\\Documents\\GitHub\\Cyclops\\data\\portfolio_returns.csv")

             1 Month  3 Months  6 Months  12 Months
Date                                               
2010-01-31       NaN       NaN       NaN        NaN
2010-02-28       NaN       NaN       NaN        NaN
2010-03-31       NaN       NaN       NaN        NaN
2010-04-30       NaN       NaN       NaN        NaN
2010-05-31       NaN       NaN       NaN        NaN
2010-06-30       NaN       NaN       NaN        NaN
2010-07-31       NaN       NaN       NaN        NaN
2010-08-31       NaN       NaN       NaN        NaN
2010-09-30       NaN       NaN       NaN        NaN
2010-10-31       NaN       NaN       NaN        NaN
2010-11-30       NaN       NaN       NaN        NaN
2010-12-31       NaN       NaN       NaN        NaN
2011-01-31       NaN       NaN       NaN        NaN
2011-02-28  0.031446       NaN       NaN        NaN
2011-03-31  0.020334       NaN       NaN        NaN
2011-04-30 -0.021650 -0.022230       NaN        NaN
2011-05-31 -0.011065 -0.016231       NaN        NaN
2011-06-30  